# NB06 머클 트리 — 하나의 뿌리로 전체를 보증하기

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB06-1: 머클 트리 — 뿌리 계산, 포함 증명 만들기, 증명 검증 (일반형: SHA-256 한 번)
import hashlib, math

def H(b: bytes) -> bytes:
    return hashlib.sha256(b).digest()

def merkle_levels(leaves):
    """잎 해시 목록에서 각 층의 해시 목록을 아래층부터 위층 순서로 돌려준다.
    홀수 개면 마지막 것을 한 번 더 써서 짝을 만든다(비트코인과 같은 규칙 — 4절 참조)."""
    levels = [leaves]
    while len(levels[-1]) > 1:
        cur = levels[-1]
        if len(cur) % 2 == 1:
            cur = cur + [cur[-1]]
        levels.append([H(cur[i] + cur[i + 1]) for i in range(0, len(cur), 2)])
    return levels

def merkle_proof(levels, index):
    """index 번째 잎의 포함 증명: (짝 해시, 짝이 오른쪽인가) 의 목록 — 「좌우 표시」가 함께 간다."""
    proof = []
    for level in levels[:-1]:
        if len(level) % 2 == 1:
            level = level + [level[-1]]
        sibling = index ^ 1                         # 짝의 자리: 짝수면 오른쪽 이웃, 홀수면 왼쪽 이웃
        proof.append((level[sibling], sibling > index))
        index //= 2
    return proof

def verify_proof(leaf, proof, root):
    h = leaf
    for sibling, sibling_is_right in proof:
        h = H(h + sibling) if sibling_is_right else H(sibling + h)
    return h == root

txs = [f"tx{i}: Alice pays Bob {i} coin".encode() for i in range(8)]
leaves = [H(t) for t in txs]
levels = merkle_levels(leaves)
root = levels[-1][0]
print("거래 8건 → 층별 해시 개수:", [len(l) for l in levels], "→ 뿌리", root.hex()[:16] + "…")

idx = 4                                            # 다섯 번째 거래(0부터 세어 4)
proof = merkle_proof(levels, idx)
print("tx4 의 포함 증명 길이:", len(proof), "개의 해시 =", 32 * len(proof), "바이트")
print("증명 검증(원본 tx4)   :", verify_proof(leaves[idx], proof, root))
print("증명 검증(변조 tx4)   :", verify_proof(H(b"tx4: Alice pays Bob 40 coin"), proof, root))

tampered = leaves[:]; tampered[2] = H(b"tx2: Alice pays Bob 200 coin")
print("tx2 를 바꾸면 뿌리가 바뀌는가:", merkle_levels(tampered)[-1][0] != root)

거래 8건 → 층별 해시 개수: [8, 4, 2, 1] → 뿌리 23db046cc041f456…
tx4 의 포함 증명 길이: 3 개의 해시 = 96 바이트
증명 검증(원본 tx4)   : True
증명 검증(변조 tx4)   : False
tx2 를 바꾸면 뿌리가 바뀌는가: True


In [ ]:
# NB06-2: 거래 수가 늘면 증명은 얼마나 느나 — 한 줄 사슬(N 단계)과 트리(log N)의 비교
import math
print(f"{'거래 수 n':>12} {'사슬 따라가기(최대 단계)':>22} {'머클 증명(해시 개수)':>20} {'증명 바이트':>10}")
for n in [1, 2, 8, 100, 1_000, 4_000, 10_000, 100_000, 1_000_000]:
    k = math.ceil(math.log2(n)) if n > 1 else 0
    print(f"{n:>12,} {n:>22,} {k:>20} {32*k:>10,}")

      거래 수 n         사슬 따라가기(최대 단계)         머클 증명(해시 개수)     증명 바이트
           1                      1                    0          0
           2                      2                    1         32
           8                      8                    3         96
         100                    100                    7        224
       1,000                  1,000                   10        320
       4,000                  4,000                   12        384
      10,000                 10,000                   14        448
     100,000                100,000                   17        544
   1,000,000              1,000,000                   20        640


In [ ]:
# NB06-3: 비트코인식 머클 트리 — 이중 SHA-256, 홀수 복제, 그리고 CVE-2012-2459 가 말하는 「같은 뿌리」
import hashlib

def H2(b: bytes) -> bytes:                         # 비트코인의 해시: SHA256(SHA256(x))
    return hashlib.sha256(hashlib.sha256(b).digest()).digest()

def bitcoin_merkle_root(txids):
    """txid 목록(내부 바이트 순서)에서 뿌리를 계산한다. 홀수면 마지막을 복제."""
    layer = list(txids)
    while len(layer) > 1:
        if len(layer) % 2 == 1:
            layer.append(layer[-1])
        layer = [H2(layer[i] + layer[i + 1]) for i in range(0, len(layer), 2)]
    return layer[0]

ids = [H2(bytes([i])) for i in range(1, 7)]        # 거래 1~6 의 가짜 txid
root_a = bitcoin_merkle_root(ids)
root_b = bitcoin_merkle_root(ids + ids[4:6])       # [1,2,3,4,5,6,5,6]
print("[1..6] 뿌리         :", root_a.hex())
print("[1..6,5,6] 뿌리     :", root_b.hex())
print("두 목록의 뿌리가 같은가:", root_a == root_b, " ← 거래 목록이 다른데 뿌리가 같다")

# 거래가 하나뿐인 블록(제네시스 블록이 그렇다)에서는 뿌리 = 그 거래의 txid
one = H2(b"only one transaction")
print("거래 1건이면 뿌리 == txid :", bitcoin_merkle_root([one]) == one, "(12장에서 진짜 데이터로 확인)")

[1..6] 뿌리         : df316ddf151a6d7f65a868df97de2168ed674371119b038bb97c9926b3f7948d
[1..6,5,6] 뿌리     : df316ddf151a6d7f65a868df97de2168ed674371119b038bb97c9926b3f7948d
두 목록의 뿌리가 같은가: True  ← 거래 목록이 다른데 뿌리가 같다
거래 1건이면 뿌리 == txid : True (12장에서 진짜 데이터로 확인)
